## INTRODUCTION
In (Orellana et al. 2016) we have used all the crystal structures of a protein to create an ensemble. What we call an ensemble in this article is a set of structures that are not chimeric, represent the whole protein rather than a single domain of it, and do not contain more than 5 consecutive nonterminal missing residues. As much as we would like to automate this process, preparation of an ensemble often requires manual intervention due to problems like, chimeric constructs or the placement of subunits clockwise vs. anticlockwise. There is also the decision to be made whether to exclude a structure that has missing residues that are not terminal.
    
While preparing the ensembles for the article, we have taken care of those problems manually and made a decision to repair the structures if it was known to represent an important state which does not have any other representatives. Altough it very much depends on the structure, whether there are other structures in a similar state to be used as template or where the missing residues are located, we recommend repairing structures with less than 6 consecutive residues missing. There is an example modeller script that can be used to repair structures. It is upto the user to fix and include them in the ensemble later. 
    
This tutorial will be using GLIC as an example. Because this was most challenging ensemble we prepared for this study, it highlights the potential issues the users likely to encounter.

## HOW TO
### Requirements
Before we get started make sure you have installed the pdbParser package. You will find the other required python packages within [github](https://github.com/ozyo/pdbParser) README.md

To determine the missing residues, pdbParser relies on sequence alignment. If you use pdbParser to prepare a start and an end structure for eBDIMs then the alignment is generated via BioPython. However if you are preparing an ensemble you must install Clustal Omega or run the extracted sequences through a multiple sequence alignment tool and save the alignment in *fasta* format. The installation of Clustal Omega can easily be done via conda. After you install it you should provide the full path for the clustalo executable if it is not already in your path.

Altough not a requirement, if you want to fix any broken structures you will need a modelling program. We have provided an example script for MODELLER.

### Obtaining PDB Files
  It can be daunting to find all the available PDB files, and the chains that actually belongs to your protein. The easiset way to do this is to go through the UniProtKB database and save the IDs. pdbParser only requires the UniProt ID and how many chains are there in a biological assembly to prepare the ensemble. 
  
  ![](img/uniprot_ini_search.png)
  
After finding the UniProt IDs you can run the commands below to gather information on the reference sequence and the available PDBs. This routine is for homomeric ensembles, where a single reference sequence is sufficient. Heteromeric structures require a different routine that is covered in the Multi UniProt Code Tutorial. The logic is similar in both that only the shared CA coordinates should be kept, any broken structures should be fixed or excluded.

There are quite a lot of files to read and write, a work directory must be created before run time. If you use "." this will use current directory for read/write. Since you could be running this notebook from any location, it is a better practice to provide a full path.

In [1]:
query='Q7NDN8' #UniPortKB ID
mer=5 #Total number of chains
cwd="./glic" #This directory must exist
#exclude=[List of PDB IDs] 
altloc="A"
#If you already know some structures you want to exclude provide them as a list. 

In [2]:
from pdbParser import prepENS as pE
#import os
#os.mkdir(cwd)
info=pE.PDBInfo(query,mer)
#to exclude pdbs already from this list: pE.PDBInfo(query,mer,exclude=[ID1,ID2])

CRITICAL:root:Cannot process PDB id 3IGQ. It does not contain complete set


PDBInfo class, during initiation, goes into the UniProt page of the given ID, and retrives the 3D structures table (as seen in figure below) and the reference sequence to be used for the alignment. For the reasons below it is much easier to prepare the ensemble via a database like UniProtKB:
* Because there is a possiblity of all the structures having the same missing residues, the reference sequence is a good control to have. 
* Within 3D structure table, only the ones with the label "X-ray" are downloaded. Currently we don't support multi-model pdb files (no NMR, sorry!).
* The relevant chains for this protein is also extracted from this table, this is much easier than reading PDB headers.

In the example above, you already see that 3IGQ structure is skipped. That is because this file had 6 subunits instead of 5. If you go into the RCSB Database and look at this structure you will see that it contains only the extracellular domain and not the whole structure. To our advantage this one had more subunits than we have set, so it is eliminated early on. However there could be other incomplete structures like this. Visualising the ensemble, inspecting the alignment manually is always a good idea!

![](img/uniprot_3dtable.png)

  After the information is gathered, info object is populated with the possible structures to be used and the reference sequence. You can reach the list from the attributes assigned to this calss. If you want to manipulate this list, add or remove structures, or change the reference sequence you can do so at this stage. Please note that all the functions in prepENS module uses and modifes this class directly. So the attiribute names are important and they should remain unaltered. The values of the attributes , on the other hand, can be manipulated to improve the ensemble.

In [3]:
print(info.__dict__.keys()) #If you want to see the attributes of the PDBInfo class ;) 
print(info.query) #UniPort ID used to initialize this class
#and it is also used for the output names.
print(info.refseq) #Contains the reference sequence. 

dict_keys(['exclude', 'query', 'mer', 'result', 'refseq', 'broken', 'seqfilename', 'residmapfilename', 'alnfasta', 'coremer', 'coreresids'])
Q7NDN8
MFPTGWRPKLSESIAASRMLWQPMAAVAVVQIGLLWFSPPVWGQDMVSPPPPIADEPLTVNTGIYLIECYSLDDKAETFKVNAFLSLSWKDRRLAFDPVRSGVRVKTYEPEAIWIPEIRFVNVENARDADVVDISVSPDGTVQYLERFSARVLSPLDFRRYPFDSQTLHIYLIVRSVDTRNIVLAVDLEKVGKNDDVFLTGWDIESFTAVVKPANFALEDRLESKLDYQLRISRQYFSYIPNIILPMLFILFISWTAFWSTSYEANVTLVVSTLIAHIAFNILVETNLPKTPYMTYTGAIIFMIYLFYFVAVIEVTVQHYLKVESQPARAASITRASRIAFPVVFLLANIILAFLFFGF


  The next command downloads the structures from the RCSB database. Depending on the number of structures this function might take a while to finish because after it downloads the files it goes through the steps below for cleaning:
  
  1. It checks for lines with TITLE to find and exclude the chimeric structures.
  2. It seperates the biological assemblies within the same PDB file
  3. It keeps only the CA coordinates, which are the only relevant atoms for comparison with eBDIMs trajectories. Obtaining only CA coordinates is also a way to clean the PDB files from bound ligands, ions, waters, etc.
  4. It extracts the sequence and the residue numbering from the CA coordinates and writes them to text files.

pdbParser relies on multiple sequence alignment to determine the missing residues. This also allows us to ignore the differences in residue numbering between different structures, and non-consecutive numbering (an indication of missing domains, insertions/deletions). The sequence of the structures are extracted from CA coordinates, rather than PDB header. Because a residue might have missing CA coordinate, despite being listed in the sequence records. 

The residue numbering information is only used to extract the residues from the the PDB file itself and it is not used in any other way in the alignment.

In the example below you will see that there are more warnings, stating that more structures will be excluded from the ensemble because they are chimeras. Since these structures cannot be used, their information is removed from the info.result dictionary automatically and those files are deleted.

If you check the output folder at this stage, you will find two text files: "_seq.txt" and "_residmap.txt". "_seq.txt" contains the sequences and "_residmap.txt" holds the corresponding residue numbering. These will be used below in the sequence alignment.

In [6]:
from importlib import reload
reload(pE)
ordch=pE.downloadPDB(info,cwd) 
#This ensures that the chain ordering for the subsequent calculations are based on the original PDB file.
info.results=ordch

2XQ3
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
2XQ4
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
2XQ5
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
2XQ6
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
2XQ7
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
2XQ8
['A' 'B' 'C' 'D' 'E'] ['A', 'B

['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T'] ['F', 'G', 'H', 'I', 'J']
['F' 'G' 'H' 'I' 'J'] ['F']
['F' 'G' 'H' 'I' 'J'] ['G']
['F' 'G' 'H' 'I' 'J'] ['H']
['F' 'G' 'H' 'I' 'J'] ['I']
['F' 'G' 'H' 'I' 'J'] ['J']
['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T'] ['K', 'L', 'M', 'N', 'O']
['K' 'L' 'M' 'N' 'O'] ['K']
['K' 'L' 'M' 'N' 'O'] ['L']
['K' 'L' 'M' 'N' 'O'] ['M']
['K' 'L' 'M' 'N' 'O'] ['N']
['K' 'L' 'M' 'N' 'O'] ['O']
['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T'] ['P', 'Q', 'R', 'S', 'T']
['P' 'Q' 'R' 'S' 'T'] ['P']
['P' 'Q' 'R' 'S' 'T'] ['Q']
['P' 'Q' 'R' 'S' 'T'] ['R']
['P' 'Q' 'R' 'S' 'T'] ['S']
['P' 'Q' 'R' 'S' 'T'] ['T']
4QH1
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
4QH4
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B

['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
6HY5
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
6HY9
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
6HYA
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
6HYR
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A']
['A' 'B' 'C' 'D' 'E'] ['B']
['A' 'B' 'C' 'D' 'E'] ['C']
['A' 'B' 'C' 'D' 'E'] ['D']
['A' 'B' 'C' 'D' 'E'] ['E']
6HYV
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C

The reason downloadPDB returned a new information about the chains is because when we extracted information from the UniProt database, we have could have lost track of the order of the chains. This is of specific important to multimeric systems like GLIC where the subunits could be placed in any order in the PDB file including non-sequential placements. Since I have not made this function an attribute of the PDBInfo class, I chose not to modify an already existing information about the class. Thus, for the following functions to work as they should we update the info.result with the new information.

### Multiple Sequence Alignment and Identification of the Shared Region
Multiple sequence alignment of the sequences together with the reference sequence is used for cleaning of the terminal residues that are not shared between the structures (most likely terminal tags, or undetermined regions). Since we use a reference sequence we can also identify structures that contain a non-terminal gap, mark them as broken and remove the assembly from the ensemble. If you have Clustal Omega installed you can proceed with the alignment generation which in the same function parses the alignment as well.

If you chose to run the alignment somwhere else, you have to do is to clean any headers that it might contain and save it in the *fasta* format. The names of the fasta identifiers must be exactly in this format: PDBID_ENSEMBLENR.pdb|CHAINID| , as given in the example file in this tutorial. The alignment should also contain a reference sequence with an identifier starting with "refseq_". The sequences extracted by pdbParser already have these identifiers, so that you don't have to make any modifications. When you have your alignment file you can, then, run the function below with the argument alnf="alignmentfile.fasta".  The alignment file must be placed where *cwd* is set to. The function will only go through the file to determine the broken chains. It does not run clustalo, so you don't have to provide the path for it or have it installed.

In [7]:
clustalopath="/Users/cattibrie_fr/miniconda2/bin/clustalo"
pE.msa(info,cwd,clustalopath)

CRITICAL:root:3UU3_1.pdb|A| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|B| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|C| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|D| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|E| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4ILA_1.pdb|E| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_2.pdb|H| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_2.pdb|I| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_3.pdb|K| cont

Noticed how I have given only the class instance as an argument? Since I have initialized the input/output filenames and created attributes within this class with the relevant data, I don't need to pass objects individually. So if you need to change names of the sequence file, for example, you can do so by changing the info.seqfilename . If you need to add or delete chains/structures you must modify both the text files and the info.result dictionary.

As you see above there are quite a few chains that have missing residues or insertions. All these identified broken chains are stored in info.broken atrribute. This list is used in the subsequent function that extracts the shared CA region. Altough the assemblies with broken chains will be skipped, the CA coordinates and the FASTA alignment information of those are kept.

Before we proceeed we can take a look at the alignment file with info.core_show. You can specificy which region of the alignment file you want to display using positions. The positions correspond to the total alignment itself, so residue number 159 is not in position 159. Alignment file will help you identify which chains you would like to repair and keep.

In [ ]:
info.core_show(cwd,positions=[220,240]) #Displays a part of the alignment
#It also adds alndata attribute to info, so that you can play around with this data frame object.

![](img/uniprot_fasta_pandas.png)

For example from the fasta alignment file, we saw that chain S in 4NPQ PDB file, assembly 2 has two missing residues. But the rest of the structures in this assembly is intact. Because there are not many "possible closed state" structures of GLIC, we decided to repair this assembly. You can use the example MODELLER script to rebuild those missing residues. After you have fixed the structure you should follow the steps below:
1. Modify the info.broken object to exclude that chain. 
2. You must also add it back to the info.coreresids and info.coremer.
3. You must rename the final, fixed file to it's original 4NPQ_4.pdb

Now that we have all the structures we want to use fixed and ready, we can move on to the next step to extract the common residues. As you can see for GLIC some files does not have the initial valine and serine residues ın the N termini. Some of the structures also have a missing residue in the C termini. This function will remove all of those based on the alignment file shown above.

In [ ]:
#%run -i 'fix_model.py' #You have to modify the contents of this file if you are working on your own example. 
#There is a possiblity of crashing the kernel with this fix_model.py script. Hence it is commented out.
# It is for MODELLER and can be run outside.
#info.broken.remove('4NPQ_4.pdb')
#Python indexing starts from 0
#chains=info.result['4NPQ'][1][3] 
#Chain informations are the 2nd element and within that list the 4th ensemble. 
#info.coremer['4NPQ_4.pdb']=chains
#info.coreresids['4NPQ_4.pdb|S|']=info.coreresids['4NPQ_4.pdb|T|'] 
#Since this is a homopentamer, I have just used the information from other chain to set the residue numbers for |S|

In [8]:
pE.getcore(info,cwd)

['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'E' 'C' 'B' 'D'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D

### Structure Alignment
Since we have extracted a common region, and kept CA coordinates only, all the structures have the same number of atoms. Regardless of the point mutations and the differences in residue numbering we can use almost any alignment tool to superimpose them. For this step you can use your favorite visualizer. In fact it is a good idea to look at the ensemble, and check each structure individually. There could still be unforseen problems. I have highlighet one of them, subunit placement. Most of the time these issues will show up in the rmsd as outliers like below.
If you want to check the rmsd you should install MDAnalysis. Pandas you should already have since it is now a requirement in pdbParser.

In [45]:
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.analysis.rms import rmsd
        
structdata={}
for mer in info.coremer:
    if mer not in info.broken:
        structdata[mer]=mda.Universe(cwd+'/correct_'+mer,cwd+'/correct_'+mer)
            
ref=structdata['4NPQ_1.pdb']
rmsds={}
for mer,uni in structdata.items():
    align.alignto(uni, ref, select="name CA", weights="mass")
    rmsds[mer]=rmsd(uni.select_atoms("name CA").positions,ref.select_atoms("name CA").positions)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(pd.DataFrame(rmsds,index=[0]).T)

                    0
5MZT_1.pdb   2.772509
5HCJ_1.pdb   2.629408
3P4W_1.pdb   2.727885
3UU5_1.pdb   2.666160
2XQ4_1.pdb   3.073073
4HFE_1.pdb   2.659620
3TLT_1.pdb   2.457261
4LMK_1.pdb  25.757103
6F0Z_1.pdb   2.690522
5MVN_1.pdb   2.705860
6F13_1.pdb   2.666561
4HFC_1.pdb   2.677175
5MUR_1.pdb   2.710846
6EMX_1.pdb   2.693557
5MZQ_1.pdb   2.711841
4HFI_1.pdb   2.664201
4ZZC_1.pdb   2.690208
5NKJ_1.pdb   2.812826
6HZ1_1.pdb   2.665286
3TLW_1.pdb   2.560939
2XQA_1.pdb   3.096392
6F15_1.pdb   2.667806
6HZW_1.pdb   2.675024
3UUB_1.pdb   2.720589
3TLU_1.pdb   2.500577
6HY9_1.pdb   2.609181
4LML_1.pdb  25.742114
5J0Z_1.pdb   2.785887
6HYX_1.pdb   2.595323
2XQ3_1.pdb   3.059601
5L47_1.pdb   2.435565
4HFD_1.pdb   2.697174
6F0U_1.pdb   2.713915
4LMJ_1.pdb  25.832274
4QH1_1.pdb   2.667375
3EAM_1.pdb   2.632891
3UU8_1.pdb   2.696381
6F0V_1.pdb   2.711589
5MZR_1.pdb   2.759251
3EI0_1.pdb   3.019476
2XQ6_1.pdb   3.064239
5MVM_1.pdb   2.717026
6F10_1.pdb   2.600747
6F0I_1.pdb   2.658377
4NPP_1.pdb

We have four outliers with different subunit ordering. Running the chain ordering routine below should fix these issues. What this code does is that it first moves the structures to the center of mass, then calculates the moment of inertia tensor. From this tensor a rotation matrix is calculated that aligns the 3rd axis from the inertia tensor to the z axis. This way all the molecules are oriented the same way and we can calculate the area of the polygon made by the center of masses of the subunits (these are the vertices of our polygon). The are is not taken to the final stage, meaning we don't take the absolute value and divide it by two. We are only interested in the sign of it to determine the subunit orientation whether it is clockwise or counterclockwise.

In [46]:
from pdbParser import chorder as cho
from importlib import reload
reload(pE)
reload(cho)
correct={ids:info.coremer[ids] for ids in info.coremer if ids not in info.broken }
reordered=cho.reorder(correct,info.mer,cwd,altloc=altloc,tag="correct_") 
#Tag is an optional argument, since we have renamed the structures as correct_ we need to provide this argument.
print("Reordered structures are:")
print(reordered)

['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'E' 'C' 'B' 'D'] ['A', 'E', 'C', 'B', 'D']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D', 'E']
['A' 'B' 'C' 'D' 'E'] ['A', 'B', 'C', 'D

4QH5_1.pdb
[-0.45324204 -0.00171    -0.89138585]
4HFB_1.pdb
[-0.44729321  0.000928   -0.8943869 ]
5HEH_1.pdb
[ 4.57087043e-01 -4.95629100e-04  8.89421829e-01]
6F11_1.pdb
[-0.45411202  0.00171568 -0.89094294]
6F7A_1.pdb
[ 0.00927091 -0.99466695  0.10272152]
4ILB_1.pdb
[-0.45424291  0.00538546 -0.89086159]
4ILC_1.pdb
[-0.44515116  0.00332206 -0.89544928]
5IUX_1.pdb
[-0.45264298  0.00156385 -0.89169047]
6HZ0_1.pdb
[-0.45744922  0.0031327  -0.88923023]
6HZ3_1.pdb
[ 4.58753124e-01 -7.57739611e-04  8.88563446e-01]
2XQ9_1.pdb
[-0.42294492 -0.00267131 -0.90615146]
5HCM_1.pdb
[-0.47823713  0.00150456 -0.87822946]
6F0R_1.pdb
[-0.45348665  0.00359719 -0.89125581]
6F0M_1.pdb
[-0.45137698  0.00186953 -0.8923314 ]
5V6O_1.pdb
[4.51554206e-01 7.45708647e-04 8.92243377e-01]
6HYA_1.pdb
[-0.47351296  0.00452384 -0.88077524]
6HY5_1.pdb
[-4.55133638e-01  5.61376169e-04 -8.90422965e-01]
4HFH_1.pdb
[-0.45608386  0.00320514 -0.88993103]
2XQ7_1.pdb
[-0.42914537 -0.00131968 -0.90323447]
6F0N_1.pdb
[-0.46037869 

In [48]:
structdata={}
for mer in info.coremer:
    if mer not in info.broken:
        try:
            structdata[mer+'_ord']=mda.Universe(cwd+'/reordered_'+mer,cwd+'/reordered_'+mer)
        except:
            structdata[mer]=mda.Universe(cwd+'/correct_'+mer,cwd+'/correct_'+mer)
            
ref=structdata['4NPQ_1.pdb']
rmsds={}
for mer,uni in structdata.items():
    align.alignto(uni, ref, select="name CA", weights="mass")
    rmsds[mer]=rmsd(uni.select_atoms("name CA").positions,ref.select_atoms("name CA").positions)
rmsds=pd.DataFrame(rmsds,index=[0]).T
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(rmsds)

                        0
5MZT_1.pdb       2.772509
5HCJ_1.pdb       2.629408
3P4W_1.pdb       2.727885
3UU5_1.pdb       2.666160
2XQ4_1.pdb       3.073073
4HFE_1.pdb       2.659620
3TLT_1.pdb       2.457261
4LMK_1.pdb      25.757103
6F0Z_1.pdb       2.690522
5MVN_1.pdb       2.705860
6F13_1.pdb       2.666561
4HFC_1.pdb       2.677175
5MUR_1.pdb       2.710846
6EMX_1.pdb       2.693557
5MZQ_1.pdb       2.711841
4HFI_1.pdb       2.664201
4ZZC_1.pdb       2.690208
5NKJ_1.pdb       2.812826
6HZ1_1.pdb       2.665286
3TLW_1.pdb       2.560939
2XQA_1.pdb       3.096392
6F15_1.pdb       2.667806
6HZW_1.pdb       2.675024
3UUB_1.pdb       2.720589
3TLU_1.pdb       2.500577
6HY9_1.pdb       2.609181
4LML_1.pdb_ord  25.809790
5J0Z_1.pdb       2.785887
6HYX_1.pdb       2.595323
2XQ3_1.pdb       3.059601
5L47_1.pdb       2.435565
4HFD_1.pdb       2.697174
6F0U_1.pdb       2.713915
4LMJ_1.pdb_ord   2.787220
4QH1_1.pdb       2.667375
3EAM_1.pdb       2.632891
3UU8_1.pdb       2.696381
6F0V_1.pdb  

Some of the ordered structures now have a lower RMSD. But there is still an outlier. Upon close inspection you will realize that the subunit ordering in 4LMK structure is actually a little more complicated. These subunits are not placed sequentially. It is actually possible to write a function to sort these subunits in the sequential ordering as well. This will be in a future update to pdbParser. For now since this is a homomeric structure we can force a new order alphabetically. After that we have all the structures ready for the ensemble. We can combine them in a big file using the simple bash commands in the last cell.

In [54]:
#alphord=[ind[0:-4] if '_ord' in ind else ind for ind in rmsds[rmsds[0]>10].index ]
#reload(cho)
#cho.forceorder(correct,cwd,info.mer,alph=True,ordlist=alphord,tag="correct_")

structdata={}
for mer in info.coremer:
    if mer not in info.broken:
        try:
            structdata[mer+'_ord2']=mda.Universe(cwd+'/reordered_2_'+mer,cwd+'/reordered_2_'+mer)
        except:
            pass
        try:
            structdata[mer+'_ord']=mda.Universe(cwd+'/reordered_'+mer,cwd+'/reordered_'+mer)
        except:
            structdata[mer]=mda.Universe(cwd+'/correct_'+mer,cwd+'/correct_'+mer)
            
ref=structdata['4NPQ_1.pdb']
rmsds={}
for mer,uni in structdata.items():
    align.alignto(uni, ref, select="name CA", weights="mass")
    rmsds[mer]=rmsd(uni.select_atoms("name CA").positions,ref.select_atoms("name CA").positions)
rmsds=pd.DataFrame(rmsds,index=[0]).T
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(rmsds)

                         0
5MZT_1.pdb        2.772509
5HCJ_1.pdb        2.629408
3P4W_1.pdb        2.727885
3UU5_1.pdb        2.666160
2XQ4_1.pdb        3.073073
4HFE_1.pdb        2.659620
3TLT_1.pdb        2.457261
4LMK_1.pdb_ord2  34.216549
4LMK_1.pdb       25.757103
6F0Z_1.pdb        2.690522
5MVN_1.pdb        2.705860
6F13_1.pdb        2.666561
4HFC_1.pdb        2.677175
5MUR_1.pdb        2.710846
6EMX_1.pdb        2.693557
5MZQ_1.pdb        2.711841
4HFI_1.pdb        2.664201
4ZZC_1.pdb        2.690208
5NKJ_1.pdb        2.812826
6HZ1_1.pdb        2.665286
3TLW_1.pdb        2.560939
2XQA_1.pdb        3.096392
6F15_1.pdb        2.667806
6HZW_1.pdb        2.675024
3UUB_1.pdb        2.720589
3TLU_1.pdb        2.500577
6HY9_1.pdb        2.609181
4LML_1.pdb_ord2  25.742114
4LML_1.pdb_ord   25.809790
5J0Z_1.pdb        2.785887
6HYX_1.pdb        2.595323
2XQ3_1.pdb        3.059601
5L47_1.pdb        2.435565
4HFD_1.pdb        2.697174
6F0U_1.pdb        2.713915
4LMJ_1.pdb_ord    2.787220
4

In [ ]:
%%bash
touch ./glic/GLIC_ensemble.pdb
for rpdb in `ls -1 ./glic/reordered_*pdb`
do
newname=`sed "/s/reordered/correct/g" <(echo $rdpb)`
mv $rpdb $newname
done

for pdb in `ls -1 ./glic/correct*pdb`
do
echo "MODEL" >> ./glic/GLIC_ensemble.pdb
cat $pdb >> ./glic/GLIC_ensemble.pdb
echo "END" >> ./glic/GLIC_ensemble.pdb
done

<table><tr>
<td> <img src="img/side.png" alt="Drawing" style="width: 100%;"/> </td>
<td> <img src="img/top.png" alt="Drawing" style="width: 100%;"/> </td>
</tr></table>

If you have any questions do not hesitate to contact us. pdbParser, ensemble preperation is a package being developed. Watch out for new features on [github](https://github.com/ozyo/pdbParser)